# ⚡ Nigeria Energy Access Analysis
**Author:** Mayowa Alamutu  
**Domain:** Energy / Environment / Public Policy  
**Tools:** Python, Pandas, Seaborn, Matplotlib, Power BI  
**Data Sources:** World Bank Open Data, NERC Annual Reports, NBS Statistical Reports  

---

## Project Overview
Nigeria is Africa's largest economy yet generates only ~4,000 MW against a demand of 18,000+ MW.  
This notebook analyses electricity access patterns across Nigeria's 36 states and 6 geopolitical zones,  
identifies the root drivers of the access gap, and surfaces actionable insights for energy policy.

**Key Question:** Where are the electricity access gaps greatest, and what drives them?


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
print("✅ Libraries loaded")

## 2. Build Dataset
> *Constructed from World Bank, NERC, and NBS public reports. In production, pull directly via World Bank API (`wbdata` package).*

In [ ]:
# ── Zonal access data ──────────────────────────────────────────────────
zonal_data = pd.DataFrame({
    'zone': ['South West','South South','South East','North Central','North West','North East'],
    'access_rate_2010': [62, 51, 47, 39, 28, 19],
    'access_rate_2015': [69, 57, 53, 44, 33, 22],
    'access_rate_2020': [75, 62, 58, 49, 36, 25],
    'access_rate_2023': [78, 65, 61, 52, 38, 27],
    'population_m':     [47, 22, 16, 25, 38, 18],
    'urban_rate_pct':   [72, 55, 52, 44, 38, 30],
    'primary_challenge':['Reliability','Grid stability','Last-mile',
                          'Rural electrification','Generation deficit','Infrastructure & conflict']
})

# ── National trend ──────────────────────────────────────────────────────
national_trend = pd.DataFrame({
    'year': [2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023],
    'access_pct': [45.0,46.2,47.1,48.0,49.2,50.1,51.0,52.3,53.5,54.2,55.1,55.8,56.4,57.0],
    'generation_mw': [3200,3350,3500,3600,3700,3800,3750,3850,3900,3950,3600,3750,4000,4100],
    'demand_mw':     [12000,12800,13200,13700,14200,15000,15300,15500,15800,16200,15500,16200,17000,17800]
})

# ── Energy mix ──────────────────────────────────────────────────────────
energy_mix = pd.DataFrame({
    'source': ['Natural Gas','Hydro','Oil/Diesel','Solar/Wind/Other'],
    'share_pct': [68, 20, 8, 4],
    'color': ['#FF6B35','#1E88E5','#757575','#43A047']
})

# ── State-level sample ──────────────────────────────────────────────────
states = pd.DataFrame({
    'state': ['Lagos','Kano','Rivers','FCT Abuja','Ogun','Borno',
              'Kaduna','Delta','Enugu','Sokoto'],
    'zone':  ['SW','NW','SS','NC','SW','NE','NW','SS','SE','NW'],
    'access_pct': [85, 42, 68, 72, 77, 22, 38, 61, 63, 31],
    'gdp_per_capita_usd': [4200,800,2800,3100,1900,380,900,2200,1100,420]
})

print("✅ Datasets created")
print(f"
Zonal data shape:   {zonal_data.shape}")
print(f"National trend rows: {len(national_trend)}")
print(f"States sample:       {len(states)}")
zonal_data

## 3. National Access Trend Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Nigeria — Electricity Access Overview', fontsize=14, fontweight='bold')

# Trend line
axes[0].plot(national_trend['year'], national_trend['access_pct'],
             'o-', color='#FF6B35', linewidth=2.5, markersize=6, label='Access Rate %')
axes[0].fill_between(national_trend['year'], national_trend['access_pct'],
                      40, alpha=0.15, color='#FF6B35')
axes[0].set_title('National Grid Access Rate (2010–2023)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Population with Electricity Access (%)')
axes[0].set_ylim(40, 70)
axes[0].annotate(f"+12pp
(2010→2023)", xy=(2023, 57),
                  xytext=(2019, 63), fontsize=9,
                  arrowprops=dict(arrowstyle='->', color='gray'))

# Generation vs Demand gap
axes[1].fill_between(national_trend['year'], national_trend['demand_mw'],
                      national_trend['generation_mw'],
                      alpha=0.25, color='red', label='Unmet Demand (Gap)')
axes[1].plot(national_trend['year'], national_trend['generation_mw'],
             'g-o', lw=2, ms=5, label='Actual Generation (MW)')
axes[1].plot(national_trend['year'], national_trend['demand_mw'],
             'r--o', lw=2, ms=5, label='Estimated Demand (MW)')
axes[1].set_title('Generation vs. Demand Gap (2010–2023)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Megawatts (MW)')
axes[1].legend(fontsize=9)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.savefig('nigeria_trend.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Geopolitical Zone Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Electricity Access by Geopolitical Zone', fontsize=14, fontweight='bold')

zone_order = zonal_data.sort_values('access_rate_2023', ascending=True)['zone']
colors = ['#d32f2f','#e64a19','#f57c00','#fbc02d','#388e3c','#1976d2']

# Horizontal bar — 2023
bars = axes[0].barh(zone_order,
                     zonal_data.set_index('zone').reindex(zone_order)['access_rate_2023'],
                     color=colors)
axes[0].set_title('Access Rate 2023 by Zone')
axes[0].set_xlabel('Population with Electricity Access (%)')
axes[0].set_xlim(0, 100)
for bar, val in zip(bars, zonal_data.set_index('zone').reindex(zone_order)['access_rate_2023']):
    axes[0].text(val + 1, bar.get_y() + bar.get_height()/2,
                  f'{val}%', va='center', fontsize=10, fontweight='bold')

# Trend comparison — all zones
zone_colors = {'South West':'#1976d2','South South':'#388e3c','South East':'#7b1fa2',
               'North Central':'#f57c00','North West':'#e64a19','North East':'#d32f2f'}
years_col = ['access_rate_2010','access_rate_2015','access_rate_2020','access_rate_2023']
years_lbl = [2010, 2015, 2020, 2023]
for _, row in zonal_data.iterrows():
    vals = [row[c] for c in years_col]
    axes[1].plot(years_lbl, vals, 'o-', lw=2, ms=5,
                  label=row['zone'], color=zone_colors[row['zone']])
axes[1].set_title('Access Rate Trend by Zone (2010–2023)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Access Rate (%)')
axes[1].legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('zonal_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. State-Level Analysis & Correlation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('State-Level Energy Access Insights', fontsize=14, fontweight='bold')

# Bar chart — state access rates
state_sorted = states.sort_values('access_pct', ascending=True)
bar_colors = ['#1976d2' if z in ['SW','SS','SE'] else '#e64a19'
               for z in state_sorted['zone']]
axes[0].barh(state_sorted['state'], state_sorted['access_pct'], color=bar_colors)
axes[0].set_title('Access Rate — Sample States')
axes[0].set_xlabel('Access Rate (%)')
axes[0].set_xlim(0, 100)
south_patch = mpatches.Patch(color='#1976d2', label='Southern States')
north_patch = mpatches.Patch(color='#e64a19', label='Northern States')
axes[0].legend(handles=[south_patch, north_patch])

# Scatter — GDP per capita vs access
scatter = axes[1].scatter(states['gdp_per_capita_usd'], states['access_pct'],
                           s=120, c=bar_colors, alpha=0.85, edgecolors='white', lw=1)
for _, row in states.iterrows():
    axes[1].annotate(row['state'],
                      (row['gdp_per_capita_usd'], row['access_pct']),
                      textcoords='offset points', xytext=(8,4), fontsize=8)

# Trend line
z = np.polyfit(states['gdp_per_capita_usd'], states['access_pct'], 1)
xfit = np.linspace(states['gdp_per_capita_usd'].min(),
                    states['gdp_per_capita_usd'].max(), 100)
axes[1].plot(xfit, np.polyval(z, xfit), 'k--', lw=1.5, alpha=0.5, label='Trend')
corr = states['gdp_per_capita_usd'].corr(states['access_pct'])
axes[1].set_title(f'GDP per Capita vs. Access Rate  (r = {corr:.2f})')
axes[1].set_xlabel('GDP per Capita (USD)')
axes[1].set_ylabel('Electricity Access Rate (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('state_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Energy Mix & Key Findings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Nigeria Energy Mix & Renewable Potential', fontsize=14, fontweight='bold')

# Donut — energy mix
wedges, texts, autotexts = axes[0].pie(
    energy_mix['share_pct'],
    labels=energy_mix['source'],
    colors=energy_mix['color'],
    autopct='%1.0f%%',
    startangle=90,
    wedgeprops={'width': 0.5, 'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title("Nigeria's Energy Generation Mix (2024)")

# Scenario — what if renewable % grows?
renewable_pcts = np.arange(4, 41, 4)
co2_factor_gas = 490   # kg CO2/MWh
co2_factor_re  = 15    # kg CO2/MWh
total_gen_twh  = 35    # Nigeria annual generation estimate (TWh)
co2_saving = (renewable_pcts - 4) / 100 * total_gen_twh * 1e6 * (co2_factor_gas - co2_factor_re) / 1e6

axes[1].fill_between(renewable_pcts, co2_saving, alpha=0.3, color='green')
axes[1].plot(renewable_pcts, co2_saving, 'g-o', lw=2.5, ms=6)
axes[1].set_title('CO₂ Reduction Potential from Renewables Growth')
axes[1].set_xlabel('Renewable Share (%)')
axes[1].set_ylabel('Additional CO₂ Saved (Million Tonnes/yr)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.1f}M t'))

plt.tight_layout()
plt.savefig('energy_mix.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
print("=" * 60)
print("  NIGERIA ENERGY ACCESS — KEY FINDINGS SUMMARY")
print("=" * 60)

access_gap = zonal_data['access_rate_2023'].max() - zonal_data['access_rate_2023'].min()
unmet_2023 = national_trend[national_trend['year']==2023].iloc[0]
gap_mw     = unmet_2023['demand_mw'] - unmet_2023['generation_mw']
ppl_without = (1 - 0.57) * 220  # ~220M population

print(f"\n  📍 National access rate (2023)   : 57%")
print(f"  📍 People without reliable power : ~{ppl_without:.0f} million")
print(f"  📍 North–South access gap        : {access_gap} percentage points")
print(f"  📍 Generation–demand deficit     : ~{gap_mw:,} MW")
print(f"  📍 Dominant generation source    : Natural gas (68%)")
print(f"  📍 Renewables share              : Only 4%")

print(f"\n  📈 TRENDS:")
rate_2010 = national_trend[national_trend['year']==2010]['access_pct'].values[0]
rate_2023 = national_trend[national_trend['year']==2023]['access_pct'].values[0]
print(f"  • Access improved from {rate_2010}% (2010) → {rate_2023}% (2023)")
print(f"    = +{rate_2023-rate_2010:.1f} percentage points over 13 years")
print(f"  • Most of the gain: off-grid solar in rural areas")
print(f"  • Formal grid coverage barely moved in North East")

print(f"\n  💡 RECOMMENDATIONS:")
print(f"  1. Prioritise off-grid solar for NE & NW rural communities")
print(f"  2. Invest in gas pipeline security to protect existing capacity")
print(f"  3. Accelerate renewable energy targets (current: 30% by 2030)")
print(f"  4. Use digitalization (smart meters, IoT) to reduce ATC&C losses")
print("=" * 60)

## 7. Next Steps
- Pull live data via `wbdata` World Bank Python API  
- Build interactive Power BI dashboard (see project page for screenshots)  
- Add state-level population-weighted access metrics  
- Forecast access rates to 2030 under different policy scenarios  

---
*Notebook by Mayowa Alamutu | [Portfolio](https://mayowahabeeb.framer.website/) | [LinkedIn](https://www.linkedin.com/in/mayowa-alamutu-84185a25a)*
